# Predicción de contratación de depósitos bancarios

## 1. Carga y descripción del dataset

Se utiliza `bank-full.csv` del conjunto [Bank Marketing de UCI](https://archive.ics.uci.edu/dataset/222/bank+marketing). El archivo contiene **45,211 registros y 17 columnas**: 16 características y la variable objetivo `y`. El objetivo del experimento es estimar qué clientes podrían contratar un depósito para priorizar llamadas futuras. El archivo se lee con separador de punto y coma (`;`).

| Variable | Tipo en Pandas | Significado |
| --- | --- | --- |
| `age` | `int64` | Edad del cliente. |
| `job` | `object` | Ocupación. |
| `marital` | `object` | Estado civil. |
| `education` | `object` | Nivel educativo. |
| `default` | `object` | Si tiene un crédito en incumplimiento. |
| `balance` | `int64` | Saldo anual promedio en euros. |
| `housing` | `object` | Si tiene préstamo de vivienda. |
| `loan` | `object` | Si tiene préstamo personal. |
| `contact` | `object` | Medio de contacto utilizado. |
| `day` | `int64` | Día del mes del último contacto. |
| `month` | `object` | Mes del último contacto. |
| `duration` | `int64` | Duración de la última llamada en segundos. |
| `campaign` | `int64` | Número de contactos durante la campaña actual. |
| `pdays` | `int64` | Días desde un contacto de una campaña anterior; `-1` indica que no hubo uno. |
| `previous` | `int64` | Número de contactos previos a esta campaña. |
| `poutcome` | `object` | Resultado de una campaña anterior. |
| `y` | `object` | Variable objetivo: `yes` si contrató el depósito; `no` si no. |

En total se cargaron **7 columnas numéricas** (`int64`) y **10 columnas de texto** (`object`), incluida la variable objetivo.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
path = Path("data/bank-full.csv")
df = pd.read_csv(path, sep=";")

print(df.shape)
display(df.head())
df.info()

In [ ]:
display(df.describe(include="all").T)

print("Distribución de la variable objetivo:")
display(df["y"].value_counts())
display(df["y"].value_counts(normalize=True).mul(100).round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df["y"].value_counts().plot.bar(ax=axes[0], title="Contratación del depósito")
axes[0].set_xlabel("¿Contrató?")

df["age"].plot.hist(bins=25, ax=axes[1], title="Distribución de edades")
axes[1].set_xlabel("Edad")

df.boxplot(column="balance", ax=axes[2])
axes[2].set_title("Valores extremos de balance")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, columna, titulo in [
    (axes[0], "housing", "Según préstamo de vivienda"),
    (axes[1], "education", "Según nivel educativo")
]:
    tasas = pd.crosstab(df[columna], df["y"], normalize="index")["yes"] * 100
    tasas.sort_values().plot.bar(ax=ax)
    ax.set_title(titulo)
    ax.set_xlabel("")
    ax.set_ylabel("Clientes que contrataron (%)")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 2. Análisis exploratorio (EDA)

Las estadísticas descriptivas muestran una edad mediana de 39 años (mínimo 18, máximo 95). El saldo `balance` tiene mediana de 448 euros y media de 1,362 euros; la diferencia, junto con el diagrama de caja, muestra una distribución con valores altos poco frecuentes.

La variable objetivo está desbalanceada: 39,922 clientes (88.30 %) no contrataron el depósito y 5,289 (11.70 %) sí lo hicieron. Por ello, la exactitud por sí sola puede dar una impresión engañosa del desempeño. En la evaluación revisaremos también precisión, recall, F1 y la matriz de confusión.

Las gráficas de relaciones muestran que la proporción observada de contratación es mayor entre clientes sin préstamo de vivienda (aproximadamente 17 %) que entre quienes sí lo tienen (aproximadamente 8 %). También es mayor en el grupo de educación `tertiary` que en `primary`. Estas diferencias describen asociaciones en la muestra; no demuestran que las características causen la contratación.

In [ ]:
columnas_texto = df.select_dtypes(include=["object", "str"]).columns
conteo = df[columnas_texto].eq("unknown").sum()

resumen_unknown = pd.DataFrame({
    "cantidad": conteo,
    "porcentaje": (conteo / len(df) * 100).round(2)
})

display(resumen_unknown[resumen_unknown["cantidad"] > 0]
        .sort_values("porcentaje", ascending=False))

In [ ]:
print("Valores nulos reales:", df.isna().sum().sum())
print("Filas exactamente duplicadas:", df.duplicated().sum())

display(
    df[["age", "balance", "campaign", "previous"]]
    .describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.99])
    .T
)


In [ ]:
# Cuantificar extremos sin asumir que son errores
for columna in ["age", "balance"]:
    q1, q3 = df[columna].quantile([0.25, 0.75])
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    extremos = ((df[columna] < limite_inferior) |
                (df[columna] > limite_superior)).sum()
    print(f"{columna}: límites IQR [{limite_inferior:.1f}, {limite_superior:.1f}], "
          f"extremos = {extremos} ({extremos / len(df) * 100:.2f} %)")

# En previous los cuartiles son cero; el IQR no distingue casos raros.
p99_previous = df["previous"].quantile(0.99)
print(f"previous: percentil 99 = {p99_previous:.1f}; "
      f"valores por encima = {(df['previous'] > p99_previous).sum()}")
display(df.nlargest(5, "previous")[["age", "previous", "pdays", "y"]])

**Decisión sobre valores extremos.** La regla de 1.5 veces el IQR marca 487 edades (1.08 %) y 4,729 saldos (10.46 %). En `previous`, donde los cuartiles son cero, 453 registros superan el percentil 99 (8.9 contactos). Estos criterios detectan valores poco frecuentes, pero no prueban errores. Conservamos las observaciones porque una edad avanzada, un saldo negativo o muchos contactos previos pueden ser datos reales. No se recortan ni eliminan filas en este baseline; la sensibilidad del escalado y del modelo a estos valores queda como limitación a revisar después.

## Entrenamiento y experimentación
El entrenamiento, la evaluación y MLflow se ejecutan desde `train.py`. Este notebook se conserva únicamente para explorar los datos. Consulte `README.md` para los comandos de reproducción.
